# WIA3003: Academic Project II
## QCNN Sigmoid Readout

Compare the existing clipped linear readout with a sigmoid readout. Keep the split, circuit, seeds and learning rate fixed. Use validation only; do not select settings on test data. Codex assisted with implementation and execution.

## 1.0 Dataset Preparation

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from pennylane import numpy as pnp
from sklearn.preprocessing import MinMaxScaler

current_folder = Path.cwd()
project_folder = (
    current_folder.parent
    if current_folder.name == "notebooks"
    else current_folder
)

split_path = project_folder / "results" / "qcnn_baseline" / "run_details.json"
split_details = json.loads(split_path.read_text(encoding="utf-8"))

data_path = (
    Path.home()
    / "Downloads"
    / "Orbital, Nasal and Craniofacial Class Pattern.xls"
)

workbook_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()
if workbook_hash != split_details["dataset_sha256"]:
    raise ValueError(
        "The workbook differs from the one used for the saved QCNN split. "
        "Check the file before reusing the row positions."
    )

df = pd.read_excel(data_path, sheet_name="Raw Data")
df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()


In [2]:
features = split_details["features"]
label_mapping = split_details["label_mapping"]

fit_indices = np.array(split_details["fit_row_positions"], dtype=int)
val_indices = np.array(split_details["validation_row_positions"], dtype=int)

fit_records = df.iloc[fit_indices]
val_records = df.iloc[val_indices]

X_fit_raw = fit_records[features].apply(pd.to_numeric, errors="raise")
X_val_raw = val_records[features].apply(pd.to_numeric, errors="raise")

y_fit_raw = fit_records["Gender"].astype("string").str.strip().map(label_mapping)
y_val_raw = val_records["Gender"].astype("string").str.strip().map(label_mapping)

assert X_fit_raw.shape == (664, 8)
assert X_val_raw.shape == (168, 8)
assert not set(fit_indices).intersection(val_indices)
assert np.isfinite(X_fit_raw.to_numpy()).all()
assert np.isfinite(X_val_raw.to_numpy()).all()
assert not y_fit_raw.isna().any()
assert not y_val_raw.isna().any()

fit_groups = set(X_fit_raw.itertuples(index=False, name=None))
val_groups = set(X_val_raw.itertuples(index=False, name=None))
shared_groups = fit_groups & val_groups

assert not shared_groups, "Identical measurements appear in both partitions."

print("Fitting:", X_fit_raw.shape)
print("Validation:", X_val_raw.shape)
print("Shared measurement groups:", len(shared_groups))


Fitting: (664, 8)
Validation: (168, 8)
Shared measurement groups: 0


In [3]:
scaler = MinMaxScaler(feature_range=(0, np.pi), clip=True)

X_fit = pnp.array(
    scaler.fit_transform(X_fit_raw),
    requires_grad=False,
)

X_val = pnp.array(
    scaler.transform(X_val_raw),
    requires_grad=False,
)

y_fit = pnp.array(y_fit_raw.to_numpy(dtype=int), requires_grad=False)
y_val = pnp.array(y_val_raw.to_numpy(dtype=int), requires_grad=False)

validation_values = X_val_raw.to_numpy()
clipped_count = (
    (validation_values < scaler.data_min_)
    | (validation_values > scaler.data_max_)
).sum()

print("Validation feature values clipped:", int(clipped_count))


Validation feature values clipped: 1


## 2.0 Circuit and Readout

The circuit still returns four expectations. Apply a sigmoid to a weighted sum of them. Initialise the logit to four times the old score minus its 0.5 offset, i.e. 4 × (old score − 0.5). This preserves the initial decision boundary and matches the slope at the boundary, but initial probabilities differ away from it. Compute binary cross-entropy directly from logits for numerical stability.

In [4]:
import time
import pennylane as qml
from sklearn.metrics import accuracy_score, f1_score, log_loss, recall_score

SEEDS = [42, 123, 2026]
BATCH_SIZE = 32
MAX_EPOCHS = 30
PATIENCE = 5
RATE = 0.01


In [5]:
n_qubits = 8
surviving_wires = [1, 3, 5, 7]

def convolution_block(parameters, first, second):
    qml.Rot(*parameters[:3], wires=first)
    qml.Rot(*parameters[3:], wires=second)
    qml.CNOT(wires=[first, second])

def convolution_layer(parameters):
    # Apply the same block across neighbouring pairs.
    for first in [0, 2, 4, 6]:
        convolution_block(parameters, first, first + 1)
    for first in [1, 3, 5]:
        convolution_block(parameters, first, first + 1)

def pooling_layer(angle):
    for source, target in [(0, 1), (2, 3), (4, 5), (6, 7)]:
        qml.CRY(angle, wires=[source, target])

device = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(device, interface="autograd", diff_method="backprop")
def qcnn_circuit(inputs, weights):
    for wire in range(n_qubits):
        qml.RY(inputs[..., wire], wires=wire)
    convolution_layer(weights[:6])
    convolution_layer(weights[6:12])
    pooling_layer(weights[12])
    return [qml.expval(qml.PauliZ(wire)) for wire in surviving_wires]




In [6]:
def readout_logit(parameters, inputs):
    z = pnp.stack(qcnn_circuit(inputs, parameters[:13]))
    return pnp.sum(parameters[13:17, None] * z, axis=0) + parameters[17]


def predict_probability(parameters, inputs, readout):
    logits = readout_logit(parameters, inputs)
    return pnp.exp(-pnp.logaddexp(0, -logits))


def binary_loss(parameters, inputs, labels, readout):
    logits = readout_logit(parameters, inputs)
    return pnp.mean(pnp.logaddexp(0, (1 - 2 * labels) * logits))


In [7]:
def train_readout(seed, readout, learning_rate):
    # Same circuit initialisation for both readouts at each seed.
    rng = np.random.default_rng(seed)
    circuit_weights = rng.uniform(-0.1, 0.1, size=13)

    if readout == "Learned readout":
        initial_weights = np.concatenate(
            [circuit_weights, np.full(4, -0.5), [0.0]]
        )
    else:
        initial_weights = circuit_weights

    parameters = pnp.array(initial_weights, requires_grad=True)
    optimiser = qml.AdamOptimizer(stepsize=learning_rate)
    shuffle_rng = np.random.default_rng(seed)

    best_loss = np.inf
    best_epoch = None
    best_parameters = None
    waiting = 0
    history = []
    start = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        order = shuffle_rng.permutation(len(X_fit))

        for offset in range(0, len(order), BATCH_SIZE):
            batch = order[offset : offset + BATCH_SIZE]

            def batch_loss(current):
                return binary_loss(
                    current, X_fit[batch], y_fit[batch], readout
                )

            parameters = optimiser.step(batch_loss, parameters)

        fitting_loss = float(binary_loss(parameters, X_fit, y_fit, readout))
        validation_loss = float(binary_loss(parameters, X_val, y_val, readout))

        if not np.isfinite([fitting_loss, validation_loss]).all():
            raise ValueError(f"Non-finite loss: {readout}, seed {seed}, epoch {epoch}")

        history.append({
            "Readout": readout,
            "Seed": seed,
            "Epoch": epoch,
            "Fitting loss": fitting_loss,
            "Validation loss": validation_loss,
        })

        if validation_loss < best_loss:
            best_loss = validation_loss
            best_epoch = epoch
            best_parameters = np.array(parameters, dtype=float, copy=True)
            waiting = 0
        else:
            waiting += 1
            if waiting >= PATIENCE:
                break

    runtime = time.perf_counter() - start

    # Score the best checkpoint, not the last training epoch.
    checkpoint = pnp.array(best_parameters, requires_grad=False)
    probabilities = np.asarray(
        predict_probability(checkpoint, X_val, readout), dtype=float
    )
    predictions = (probabilities >= 0.5).astype(int)
    labels = np.asarray(y_val, dtype=int)

    result = {
        "Readout": readout,
        "Seed": seed,
        "Best epoch": best_epoch,
        "Epochs run": len(history),
        "Validation macro F1": f1_score(
            labels, predictions, average="macro", zero_division=0
        ),
        "Validation accuracy": accuracy_score(labels, predictions),
        "Validation loss": log_loss(labels, probabilities, labels=[0, 1]),
        "Runtime (s)": runtime,
        "Female recall": recall_score(labels, predictions, pos_label=0, zero_division=0),
        "Male recall": recall_score(labels, predictions, pos_label=1, zero_division=0),
    }

    return result, history, best_parameters


## 3.0 Train and Compare

Compare validation macro F1 seed by seed. Use minimum validation loss to select each checkpoint. The clipped reference comes from notebook 06.

In [8]:
output = project_folder / "results" / "qcnn_sigmoid_readout"
output.mkdir(parents=True, exist_ok=True)
reference = pd.read_csv(project_folder / "results" / "qcnn_learning_rate" / "validation_runs.csv")
reference = reference.loc[np.isclose(reference["Learning rate"], RATE) & reference["Readout"].eq("Learned readout")].copy()
assert sorted(reference["Seed"].tolist()) == sorted(SEEDS)
reference["Readout"] = "Clipped linear"

results = []
for seed in SEEDS:
    print(f"Training sigmoid readout: seed {seed}", flush=True)
    result, history, weights = train_readout(seed, "Learned readout", RATE)
    result["Readout"] = "Sigmoid"
    result["Learning rate"] = RATE
    results.append(result)
    np.save(output / f"seed_{seed}_weights.npy", weights)
    pd.DataFrame(history).to_csv(output / f"seed_{seed}_history.csv", index=False)
    pd.DataFrame(results).to_csv(output / "sigmoid_runs.csv", index=False)
    print(f"Seed {seed}: macro F1 {result['Validation macro F1']:.4f}, accuracy {result['Validation accuracy']:.4f}, epoch {result['Best epoch']}", flush=True)

comparison = pd.concat([reference, pd.DataFrame(results)], ignore_index=True)
metrics = ["Validation macro F1", "Validation accuracy", "Validation loss"]
summary = comparison.groupby("Readout")[metrics].agg(["mean", "std"])
paired = comparison.pivot(index="Seed", columns="Readout", values="Validation macro F1")
paired["Sigmoid minus clipped"] = paired["Sigmoid"] - paired["Clipped linear"]
comparison.to_csv(output / "validation_runs.csv", index=False)
summary.to_csv(output / "validation_summary.csv")
paired.to_csv(output / "paired_macro_f1.csv")
display(summary.round(4))
display(paired.round(4))


Training sigmoid readout: seed 42
Seed 42: macro F1 0.7326, accuracy 0.7381, epoch 29
Training sigmoid readout: seed 123
Seed 123: macro F1 0.7295, accuracy 0.7381, epoch 29
Training sigmoid readout: seed 2026
Seed 2026: macro F1 0.7102, accuracy 0.7143, epoch 29


Validation macro F1         Validation accuracy         Validation loss        
                              mean     std                mean     std            mean     std
Readout                                                                                       
Clipped linear              0.7159  0.0035              0.7242  0.0069          0.5146  0.0009
Sigmoid                     0.7241  0.0122              0.7302  0.0137          0.5016  0.0011

Readout,Clipped linear,Sigmoid,Sigmoid minus clipped
Seed,,,
42,0.7139,0.7326,0.0187
123,0.7139,0.7295,0.0156
2026,0.7200,0.7102,-0.0098
